# Treasury Futures Curve Momentum & Mean Reversion — Research Pipeline
Ingest Bloomberg CSV exports, validate, build continuous futures and features,
run vol-scaled backtests with costs and no lookahead, and export a macro regime
series. Runs on whatever CSVs are present; strategies whose inputs are missing
are skipped with a logged warning (so it works on the test slice and scales to
the full 2010+ pull unchanged).

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
pd.set_option("display.width", 120)

# Defaults are repo-relative; override with the BBG_CSV / BBG_RESULTS env vars.
# Resolves the repo root whether the kernel starts in the repo root or in notebooks/.
_CWD = os.getcwd()
REPO_ROOT = os.path.dirname(_CWD) if os.path.basename(_CWD) == "notebooks" else _CWD
DATA_DIR = os.environ.get("BBG_CSV", os.path.join(REPO_ROOT, "data_raw"))
RESULTS_DIR = os.environ.get("BBG_RESULTS", os.path.join(REPO_ROOT, "results"))
os.makedirs(RESULTS_DIR, exist_ok=True)

MONTH_CODE = {"F":1,"G":2,"H":3,"J":4,"K":5,"M":6,"N":7,"Q":8,"U":9,"V":10,"X":11,"Z":12}
ROOTS = ["TU","FV","TY","UXY","US","WN"]        # 2y,5y,10y,ultra10,bond,ultrabond
CASH_TENORS = {"USGG3M":0.25,"USGG6M":0.5,"USGG12M":1,"USGG2YR":2,"USGG3YR":3,
               "USGG5YR":5,"USGG7YR":7,"USGG10YR":10,"USGG20YR":20,"USGG30YR":30}
LOG = []
def log(msg): LOG.append(msg); print(msg)

## 1. Loaders (schema-aware, quirk-handling)

In [ ]:
def _find(name):
    """Locate a CSV by security stem, e.g. 'TY1_Comdty' or 'USGG2YR_Index'.
    Excludes _STATIC and _DV01 side files unless the query explicitly asks for one."""
    hits = glob.glob(os.path.join(DATA_DIR, name + "*.csv"))
    if "STATIC" not in name:
        hits = [h for h in hits if "_STATIC" not in os.path.basename(h)]
    if "DV01" not in name:
        hits = [h for h in hits if "_DV01" not in os.path.basename(h)]
    return hits[0] if hits else None

def load_ts(stem):
    """Time-series file: Date + fields. Drops junk trailing rows (null settle/px)."""
    p = _find(stem)
    if not p: return None
    df = pd.read_csv(p)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).set_index("Date").sort_index()
    for c in df.columns:
        if c != "FUT_CUR_GEN_TICKER":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    price_col = "PX_SETTLE" if "PX_SETTLE" in df.columns else "PX_LAST"
    if price_col in df.columns:               # drop trailing rows with no price (holiday / today)
        last_valid = df[price_col].last_valid_index()
        if last_valid is not None:
            df = df.loc[:last_valid]
    return df

def load_static(stem):
    """Static chain file. Parses day-first date text and validates month code."""
    p = _find(stem + "_STATIC")
    if p is None:
        # exports are written as <root>1_Comdty_STATIC.csv; accept that too
        hits = sorted(glob.glob(os.path.join(DATA_DIR, stem + "*_STATIC*.csv")))
        p = hits[0] if hits else None
    if not p: return None
    df = pd.read_csv(p)
    date_cols = [c for c in df.columns if c in
                 ("LAST_TRADEABLE_DT","FUT_NOTICE_FIRST","FUT_DLV_DT_FIRST","FUT_DLV_DT_LAST")]
    for c in date_cols:
        df[c] = pd.to_datetime(df[c].astype(str), dayfirst=True, errors="coerce")
    # validate delivery month against the ticker month code (e.g. TYM82 -> M -> June)
    def month_code(t):
        s = str(t).split()[0]
        for ch in s[::-1]:
            if ch in MONTH_CODE: return MONTH_CODE[ch]
        return np.nan
    if "FUT_DLV_DT_FIRST" in df.columns:
        code_m = df["CONTRACT"].map(month_code)
        got_m  = df["FUT_DLV_DT_FIRST"].dt.month
        bad = int(((code_m == code_m) & (got_m == got_m) & (code_m != got_m)).sum())
        if bad: log(f"  [WARN] {stem}: {bad} contracts where delivery month != ticker code "
                    f"(date-locale corruption not fully repaired)")
        else:   log(f"  [ok] {stem}: delivery months consistent with ticker codes")
    return df

# discover what's available
cash   = {k: load_ts(k) for k in CASH_TENORS if load_ts(k) is not None}
fut_ts = {r: load_ts(f"{r}1") for r in ROOTS if load_ts(f"{r}1") is not None}
statics= {r: load_static(f"{r}1") for r in ROOTS if load_static(f"{r}1") is not None}
move   = load_ts("MOVE")
vix    = load_ts("VIX")
log(f"Loaded cash tenors: {sorted(cash)}")
log(f"Loaded futures roots: {sorted(fut_ts)}")

## 2. Validation report

In [ ]:
def validate(name, df, col):
    d = df[col].dropna()
    if len(d) == 0: return
    ch = d.diff().abs()
    i = ch.idxmax() if ch.notna().any() else None
    gap = d.index.to_series().diff().dt.days
    biggaps = int((gap > 4).sum())
    print(f"{name:16s} n={len(d):5d}  {d.index.min().date()}..{d.index.max().date()}  "
          f"NA={int(df[col].isna().sum()):3d}  gaps>4d={biggaps:2d}  "
          f"maxΔ={ch.max():.4f}@{i.date() if i is not None else '-'}")

print("== cash =="); [validate(k, v, "PX_LAST") for k, v in cash.items()]
print("== futures (settle) =="); [validate(k, v, "PX_SETTLE") for k, v in fut_ts.items()]
if move is not None: print("== vol =="); validate("MOVE", move, "PX_LAST")

## 3. Master calendar + cash panel
Inner-join everything onto the front-10y futures trading calendar (kills holiday
phantom rows in the cash indices automatically).

In [ ]:
cal_src = fut_ts.get("TY") if "TY" in fut_ts else (next(iter(fut_ts.values())) if fut_ts else None)
if cal_src is None: raise RuntimeError("No futures series present; cannot build calendar.")
calendar = cal_src.index

yields = pd.DataFrame({k: v["PX_LAST"] for k, v in cash.items()}).reindex(calendar).dropna(how="all")
log(f"Cash panel: {yields.shape[0]} rows x {yields.shape[1]} tenors on futures calendar")

# curve features (only those whose legs exist)
feat = pd.DataFrame(index=yields.index)
def have(*t): return all(x in yields for x in t)
if have("USGG2YR","USGG10YR"): feat["s2s10"] = yields["USGG10YR"] - yields["USGG2YR"]
if have("USGG5YR","USGG30YR"): feat["s5s30"] = yields["USGG30YR"] - yields["USGG5YR"]
if have("USGG3M","USGG10YR"):  feat["s3m10"] = yields["USGG10YR"] - yields["USGG3M"]
if have("USGG2YR","USGG5YR","USGG10YR"):
    feat["fly2510"] = 2*yields["USGG5YR"] - yields["USGG2YR"] - yields["USGG10YR"]
# PCA needs a fuller curve
if yields.shape[1] >= 5:
    z = (yields - yields.mean()) / yields.std()
    z = z.dropna()
    U, S, Vt = np.linalg.svd(z.values, full_matrices=False)
    for k in range(3): feat.loc[z.index, f"pc{k+1}"] = U[:, k] * S[k]
    log("PCA factors 1-3 computed")
else:
    log(f"[skip] PCA: only {yields.shape[1]} tenors (<5)")

## 4. Continuous back-adjusted futures
Use the front generic's settle and FUT_CUR_GEN_TICKER to flag rolls, then
difference-back-adjust to remove the calendar-spread jump at each roll.

In [ ]:
def continuous(df):
    s = df["PX_SETTLE"].copy()
    if "FUT_CUR_GEN_TICKER" in df.columns:
        gt = df["FUT_CUR_GEN_TICKER"].astype(str)
        roll = gt.ne(gt.shift()) & gt.shift().notna()
    else:
        roll = pd.Series(False, index=s.index)
    prev = s.shift()
    # settle gap introduced on each roll day (contract switch, not a real P&L move)
    gaps = pd.Series(np.where(roll, s - prev, 0.0), index=s.index)
    # difference back-adjust: subtract the sum of all *future* roll gaps from each level,
    # so historical prices line up with the current contract and roll jumps vanish.
    future_gaps = gaps[::-1].cumsum()[::-1].shift(-1).fillna(0.0)
    cont = s - future_gaps
    ret = cont.diff() / cont.shift().abs()
    return pd.DataFrame({"cont": cont, "ret": ret}), int(roll.sum())

cont = {}
for r, df in fut_ts.items():
    c, n = continuous(df)
    cont[r] = c
    log(f"{r}: {n} rolls detected, continuous series {c['cont'].notna().sum()} pts")

rets = pd.DataFrame({r: cont[r]["ret"] for r in cont}).reindex(calendar)

## 5. Signals & backtest engine
Monthly rebalance, signals from t-1, execute at t settle, vol-scaled to 10%,
cost = 1 tick/contract/side. No lookahead.

In [ ]:
TARGET_VOL = 0.10
def sharpe(x):
    x = x.dropna();  return np.sqrt(252)*x.mean()/x.std() if x.std()>0 else np.nan
def maxdd(x):
    c = (1+x.fillna(0)).cumprod();  return (c/c.cummax()-1).min()
def ann_vol(x): return np.sqrt(252)*x.dropna().std()
def rebalance_mask(idx):  # last business day of month
    return pd.Series(idx, index=idx).groupby([idx.year, idx.month]).transform("max") == idx

def run_strategy(signal, ret, cost_per_unit=0.0, min_history=126, max_lev=3.0):
    """signal: DataFrame of target (pre-vol-scale) weights; ret: matching returns.
    Vol-scale to target only after min_history days, and cap leverage at max_lev
    so a tiny early-sample realized vol can't blow the book up."""
    common = ret.dropna(how="all").index
    signal = signal.reindex(common).ffill()
    ret = ret.reindex(common)
    reb = rebalance_mask(common)
    pos = signal.where(reb).ffill().shift(1)          # decide t-1, hold
    port_raw = (pos * ret).sum(axis=1)
    realized = port_raw.rolling(63).std() * np.sqrt(252)
    scale = (TARGET_VOL / realized).clip(upper=max_lev).shift(1)
    scale[:min_history] = np.nan                      # don't trade until vol is estimable
    scale = scale.fillna(0.0)
    pos = pos.mul(scale, axis=0)
    gross = (pos * ret).sum(axis=1)
    turnover = pos.diff().abs().sum(axis=1)
    net = gross - turnover * cost_per_unit
    return pd.DataFrame({"gross": gross, "net": net, "turnover": turnover})

def report(name, res):
    n = res["net"]
    print(f"{name:24s} Sharpe(net)={sharpe(n):5.2f}  vol={ann_vol(n):.3f}  "
          f"maxDD={maxdd(n):6.1%}  turnover/yr={res['turnover'].sum()/ (len(n)/252):.1f}")
    return n

# tick value fraction for cost: 1 tick / price ~ contribution; use per-root tick from static
def tick_frac(r):
    st = statics.get(r)
    if st is not None and "FUT_TICK_SIZE" in st and fut_ts[r]["PX_SETTLE"].notna().any():
        return float(st["FUT_TICK_SIZE"].iloc[-1]) / fut_ts[r]["PX_SETTLE"].dropna().iloc[-1]
    return 0.0

### 5a. Baseline: duration (time-series) momentum

In [ ]:
curves = {}
if rets.shape[1] >= 1:
    mom = np.sign(rets.rolling(63).sum())        # 3m TS momentum
    costs = np.mean([tick_frac(r) for r in rets.columns]) if rets.shape[1] else 0
    res = run_strategy(mom, rets, cost_per_unit=costs)
    curves["DurationMomentum"] = report("Duration momentum", res)
else:
    log("[skip] duration momentum: no continuous returns")

### 5b-5d. DV01-neutral curve trades (mean-reversion)
The baseline momentum signal is wrong for the curve: slopes and curvature are
mean-reverting, not trending (a naive 2s10s momentum book runs a large negative
Sharpe with a duration beta of ~0, i.e. it is correctly hedged but on the wrong
side of the signal). So curve trades use a mean-reversion signal: fade the spread
when it is stretched vs its own 1y history. Momentum is retained only for outright
duration. DV01 uses Bloomberg's real forward risk (CONVENTIONAL_CTD_FORWARD_FRSK)
per contract per day when that column is present, and falls back to the rolling
regression-beta proxy otherwise. The hedge ratio is dv01_long / dv01_short.
CAVEAT: the momentum-vs-reversion choice for the curve is economically motivated
(levels trend, spreads revert) but should still be confirmed out-of-sample before
any hard performance claim; a single full-sample sign is not proof.

In [ ]:
def rolling_dv01_beta(root, tenor, win=252):
    if root not in cont or tenor not in yields: return None
    dy = yields[tenor].diff()
    d = pd.concat([cont[root]["ret"], dy], axis=1).dropna()
    d.columns = ["fr", "dy"]
    cov = d["fr"].rolling(win).cov(d["dy"]); var = d["dy"].rolling(win).var()
    return (cov / var).reindex(yields.index)

DV01_FIELD = "CONVENTIONAL_CTD_FORWARD_FRSK"
def real_dv01(root):
    """Daily forward DV01. Prefers the separate {root}1_Comdty_DV01.csv file
    (single-field pull); falls back to an in-TS-file column if present; else None."""
    # 1) separate DV01 side file
    p = _find(f"{root}1_Comdty_DV01")
    if p is not None:
        d = pd.read_csv(p)
        d["Date"] = pd.to_datetime(d["Date"], errors="coerce")
        d = d.dropna(subset=["Date"]).set_index("Date").sort_index()
        if DV01_FIELD in d.columns:
            s = pd.to_numeric(d[DV01_FIELD], errors="coerce").reindex(calendar)
            if s.notna().sum() > 252: return s
    # 2) fallback: column inside the futures TS file
    df = fut_ts.get(root)
    if df is not None and DV01_FIELD in df.columns:
        s = pd.to_numeric(df[DV01_FIELD], errors="coerce").reindex(calendar)
        if s.notna().sum() > 252: return s
    return None

USING_REAL_DV01 = any(real_dv01(r) is not None for r in cont)
log(f"DV01 source: {'real forward risk (Bloomberg)' if USING_REAL_DV01 else 'regression-beta proxy'}")

def reversion_z(series, win=252):
    """Fade extremes: +1 when the spread is depressed, -1 when stretched."""
    z = (series - series.rolling(win).mean()) / series.rolling(win).std()
    return (-z.clip(-3, 3)).shift(1)                     # lag 1: past data only

def hedge_ratio(short_root, short_tenor, long_root, long_tenor):
    """Contracts of the short leg per 1 long leg for DV01 neutrality.
    Prefers the real forward DV01; falls back to the beta proxy per leg."""
    ds, dl = real_dv01(short_root), real_dv01(long_root)
    if ds is not None and dl is not None:
        return (dl / ds).shift(1).clip(0.1, 10)          # dv01_long / dv01_short
    bs = rolling_dv01_beta(short_root, short_tenor)
    bl = rolling_dv01_beta(long_root,  long_tenor)
    if bs is None or bl is None: return None
    return (bl / bs).abs().shift(1).clip(0.1, 10)

# --- 5b. 2s10s: long-end (TY) vs short-end (TU), sized to fade the slope ---
if {"TU","TY"} <= set(cont) and have("USGG2YR","USGG10YR"):
    h = hedge_ratio("TU","USGG2YR","TY","USGG10YR")
    s = reversion_z(feat["s2s10"])                       # >0 => curve too flat => position for steepening
    sig = pd.DataFrame({"TY": s, "TU": -s * h})
    res = run_strategy(sig, rets[["TY","TU"]],
                       cost_per_unit=np.mean([tick_frac("TU"), tick_frac("TY")]))
    curves["2s10s_reversion"] = report("2s10s reversion", res)
else:
    log("[skip] 2s10s: need TU,TY + 2Y,10Y")

# --- 5c. 5s30s: US vs FV ---
if {"FV","US"} <= set(cont) and have("USGG5YR","USGG30YR"):
    h = hedge_ratio("FV","USGG5YR","US","USGG30YR")
    s = reversion_z(feat["s5s30"])
    sig = pd.DataFrame({"US": s, "FV": -s * h})
    res = run_strategy(sig, rets[["US","FV"]],
                       cost_per_unit=np.mean([tick_frac("FV"), tick_frac("US")]))
    curves["5s30s_reversion"] = report("5s30s reversion", res)
else:
    log("[skip] 5s30s: need FV,US + 5Y,30Y")

# --- 5d. 2s5s10s butterfly: belly (FV) vs wings (TU,TY), fade curvature ---
if {"TU","FV","TY"} <= set(cont) and "fly2510" in feat:
    hb = hedge_ratio("FV","USGG5YR","TY","USGG10YR")     # TY wing per belly
    ha = hedge_ratio("FV","USGG5YR","TU","USGG2YR")      # TU wing per belly
    s = reversion_z(feat["fly2510"])
    sig = pd.DataFrame({"FV": 2*s, "TU": -s*ha, "TY": -s*hb})
    res = run_strategy(sig, rets[["FV","TU","TY"]],
                       cost_per_unit=np.mean([tick_frac(r) for r in ("FV","TU","TY")]))
    curves["2s5s10s_fly"] = report("2s5s10s butterfly", res)
else:
    log("[skip] butterfly: need TU,FV,TY + curve legs")

## 6. Diebold–Mariano test vs baseline (HAC / Newey–West)
The naive paired t-test assumes i.i.d. return differences, which is wrong here:
monthly rebalancing means a position is held for ~21 trading days, so daily
returns within a holding period are strongly autocorrelated and the naive
standard error is understated. We use the Diebold–Mariano statistic on the
return differential with a Newey–West (Bartlett-kernel) long-run variance to
correct for that autocorrelation, a lag long enough to span the monthly holding,
and the Harvey–Leybourne–Newbold (1997) small-sample correction with Student-t
critical values. Loss differential = difference in daily net returns, so a
positive statistic means the alternative out-returns the baseline.

In [ ]:
from scipy import stats

def newey_west_lrv(d, lag):
    """Long-run variance of a series via the Bartlett-kernel HAC estimator."""
    d = np.asarray(d, float); n = len(d); dm = d - d.mean()
    g0 = np.dot(dm, dm) / n
    lrv = g0
    for k in range(1, lag + 1):
        w = 1.0 - k / (lag + 1.0)                     # Bartlett weight
        gk = np.dot(dm[k:], dm[:-k]) / n
        lrv += 2.0 * w * gk
    return max(lrv, 1e-18)

def dm_test(base_ret, alt_ret, lag=None):
    """Diebold–Mariano test, HAC variance + HLN small-sample correction.
    Returns (statistic, two-sided p-value, annualised mean edge, lag, n)."""
    d = (alt_ret - base_ret).dropna()
    n = len(d)
    if n < 30 or d.std() == 0:
        return np.nan, np.nan, np.nan, 0, n
    if lag is None:                                   # cover the ~21-day holding period
        lag = max(21, int(np.floor(4 * (n / 100.0) ** (2.0 / 9.0))))
    lrv = newey_west_lrv(d.values, lag)
    stat = d.mean() / np.sqrt(lrv / n)
    h = 1                                             # 1-step loss differential
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)   # HLN factor (~1 for h=1)
    stat *= hln
    p = 2.0 * (1.0 - stats.t.cdf(abs(stat), df=n - 1))
    return stat, p, d.mean() * 252, lag, n

def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""

base = curves.get("DurationMomentum")
if base is not None:
    print(f"{'strategy vs baseline':22s} {'DM stat':>8s} {'p-value':>8s} {'ann.edge':>9s}  lag")
    for k, v in curves.items():
        if k == "DurationMomentum": continue
        s, p, edge, lag, n = dm_test(base, v)
        print(f"{k:22s} {s:8.2f} {p:8.3f} {edge:8.1%}{stars(p):>3s}  {lag}")
    print("(H0: equal mean net return vs baseline; * / ** / *** = 10/5/1% two-sided)")

## 7. Macro regime filter + export

In [ ]:
reg = pd.DataFrame(index=calendar)
if move is not None:
    m = move["PX_LAST"].reindex(calendar).ffill()
    reg["move_z"] = (m - m.rolling(252).mean())/m.rolling(252).std()
if "s2s10" in feat:
    reg["slope"] = feat["s2s10"].reindex(calendar)
    reg["slope_chg3m"] = feat["s2s10"].diff(63).reindex(calendar)
# simple 3-state regime: risk-off if MOVE z>1, steepening/flattening by slope change sign
def classify(row):
    if "move_z" in row and row.get("move_z",0) > 1: return "stress"
    if row.get("slope_chg3m",0) > 0: return "steepening"
    return "flattening"
reg["regime"] = reg.apply(classify, axis=1)
reg.to_csv(os.path.join(RESULTS_DIR, "macro_regime.csv"))
log(f"Regime series exported: {reg['regime'].value_counts().to_dict()}")

## 8. Equity curves

In [ ]:
if curves:
    fig, ax = plt.subplots(figsize=(9,4))
    for k, v in curves.items():
        (1+v.fillna(0)).cumprod().plot(ax=ax, label=k)
    ax.legend(); ax.set_title("Net cumulative (vol-scaled, after costs)"); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR,"equity_curves.png"), dpi=110)
    print("saved equity_curves.png")

print("\n--- LOG ---"); [print(x) for x in LOG]